In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import xarray as xr
import rasterio

from rasterio.crs import CRS
from rasterio.transform import Affine
from rasterio.warp import reproject, Resampling

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from IPython.display import display

import geopandas as gpd

In [ ]:
PROJECT_DIR = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_DIR = PROJECT_DIR / "datasets"
VNP46_DIR = DATA_DIR / "VNP46"
PROCESSED_DIR = VNP46_DIR / "processed"

A1_ZARR_PATH = PROCESSED_DIR / "Haiyan_VNP46A1.zarr"
A2_ZARR_PATH = PROCESSED_DIR / "Haiyan_VNP46A2.zarr"

# Check the two locations previously used for GHSL.
GHSL_CANDIDATES = [
    VNP46_DIR / "GHSL_SMOD_E2015.tif",
    DATA_DIR / "ghsl" / "GHSL_SMOD_E2015.tif",
]

GHSL_PATH = next(
    (
        path
        for path in GHSL_CANDIDATES
        if path.exists()
    ),
    GHSL_CANDIDATES[0],
)

OUTPUT_DIR = PROJECT_DIR / "outputs" / "haiyan_diagnostics"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

EVENT_DATE = pd.Timestamp("2013-11-08")
ANALYSIS_START = EVENT_DATE - pd.Timedelta(days=180)
ANALYSIS_END = EVENT_DATE + pd.Timedelta(days=365)

# GHSL G7: dense urban clusters and urban centres.
GHSL_CLASSES = (23, 30)

# Fixed pre-Haiyan signal-mask requirements.
BASELINE_MIN_OBSERVED_DAYS = 10
BASELINE_MIN_RADIANCE = 0.0

# Used only for the phase-composite diagnostic.
PHASE_COMPLETENESS_THRESHOLD = 50.0

PHASE_WINDOWS = [
    {
        "phase": "Baseline",
        "start_day": -180,
        "end_day": -1,
    },
    {
        "phase": "Shock",
        "start_day": 0,
        "end_day": 30,
    },
    {
        "phase": "Early recovery",
        "start_day": 31,
        "end_day": 90,
    },
    {
        "phase": "Late recovery",
        "start_day": 91,
        "end_day": 180,
    },
    {
        "phase": "Long-term",
        "start_day": 181,
        "end_day": 365,
    },
]

PHASE_ORDER = [
    definition["phase"]
    for definition in PHASE_WINDOWS
]

PHASE_COLORS = {
    "Baseline": "#475569",
    "Shock": "#DC2626",
    "Early recovery": "#EA580C",
    "Late recovery": "#D97706",
    "Long-term": "#16A34A",
}

PHASE_FILL_COLORS = {
    "Baseline": "rgba(71,85,105,0.07)",
    "Shock": "rgba(220,38,38,0.09)",
    "Early recovery": "rgba(234,88,12,0.07)",
    "Late recovery": "rgba(217,119,6,0.06)",
    "Long-term": "rgba(22,163,74,0.06)",
}

MQF_COLORS = {
    "MQF = 0": "#16A34A",
    "MQF = 1": "#F59E0B",
    "MQF ≥ 2": "#DC2626",
    "No retrieval": "#CBD5E1",
}

HAIYAN_COLOR = "#2563EB"

for required_path in [
    A1_ZARR_PATH,
    A2_ZARR_PATH,
    GHSL_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required input not found:\n{required_path.resolve()}"
        )

print("VNP46A1:", A1_ZARR_PATH)
print("VNP46A2:", A2_ZARR_PATH)
print("GHSL:", GHSL_PATH)
print(
    "Analysis period:",
    ANALYSIS_START.strftime("%Y-%m-%d"),
    "to",
    ANALYSIS_END.strftime("%Y-%m-%d"),
)


In [ ]:
from pathlib import Path
import geopandas as gpd

REGION_SHAPEFILE = Path(
    "./datasets/boundaries/regions/Regions.shp"
)

region_boundary = gpd.read_file(
    REGION_SHAPEFILE
).to_crs(
    "EPSG:4326"
)

display(region_boundary)

In [ ]:
def parse_date_values(values):
    values = np.asarray(values)

    if np.issubdtype(values.dtype, np.datetime64):
        return pd.DatetimeIndex(
            pd.to_datetime(values)
        ).normalize()

    date_strings = pd.Series(
        values.astype(str)
    ).str.strip()

    if date_strings.str.fullmatch(r"\d{8}").all():
        parsed = pd.to_datetime(
            date_strings,
            format="%Y%m%d",
            errors="raise",
        )
    else:
        parsed = pd.to_datetime(
            date_strings,
            errors="raise",
        )

    return pd.DatetimeIndex(parsed).normalize()


def normalise_date_axis(dataset, product_name):
    """
    Notebook 00 stored observations along `processed`, with dates
    stored in the one-dimensional `date` variable.
    """

    if "date" in dataset.dims:
        observation_dim = "date"
        raw_dates = dataset["date"].values

    elif "date" in dataset.variables:
        date_variable = dataset["date"]

        if len(date_variable.dims) != 1:
            raise ValueError(
                f"{product_name}: `date` must be one-dimensional; "
                f"found dimensions {date_variable.dims}."
            )

        observation_dim = date_variable.dims[0]
        raw_dates = date_variable.values

    elif "time" in dataset.dims:
        observation_dim = "time"
        raw_dates = dataset["time"].values

    else:
        raise KeyError(
            f"{product_name}: no usable acquisition-date variable. "
            f"Dimensions: {dict(dataset.sizes)}. "
            f"Variables: {list(dataset.variables)}"
        )

    parsed_dates = parse_date_values(raw_dates)

    if parsed_dates.isna().any():
        raise ValueError(
            f"{product_name}: at least one acquisition date "
            "could not be parsed."
        )

    dataset = dataset.assign_coords(
        date=(
            observation_dim,
            parsed_dates.to_numpy(
                dtype="datetime64[ns]"
            ),
        )
    )

    if observation_dim != "date":
        dataset = dataset.swap_dims(
            {
                observation_dim: "date",
            }
        )

    dataset = dataset.sortby("date")

    date_index = pd.DatetimeIndex(
        dataset["date"].values
    )

    duplicate_dates = date_index[
        date_index.duplicated(
            keep=False
        )
    ]

    if len(duplicate_dates) > 0:
        duplicate_text = ", ".join(
            pd.Index(
                duplicate_dates.strftime(
                    "%Y-%m-%d"
                )
            )
            .unique()
            .tolist()[:10]
        )

        raise ValueError(
            f"{product_name}: duplicate dates remain after "
            f"Notebook 00 processing: {duplicate_text}"
        )

    return dataset


vnp46a1 = normalise_date_axis(
    xr.open_zarr(A1_ZARR_PATH),
    "VNP46A1",
)

vnp46a2 = normalise_date_axis(
    xr.open_zarr(A2_ZARR_PATH),
    "VNP46A2",
)

print("VNP46A1 dimensions:", dict(vnp46a1.sizes))
print("VNP46A2 dimensions:", dict(vnp46a2.sizes))

print(
    "VNP46A1 dates:",
    pd.Timestamp(
        vnp46a1["date"].min().item()
    ).strftime("%Y-%m-%d"),
    "to",
    pd.Timestamp(
        vnp46a1["date"].max().item()
    ).strftime("%Y-%m-%d"),
)

print(
    "VNP46A2 dates:",
    pd.Timestamp(
        vnp46a2["date"].min().item()
    ).strftime("%Y-%m-%d"),
    "to",
    pd.Timestamp(
        vnp46a2["date"].max().item()
    ).strftime("%Y-%m-%d"),
)


In [ ]:
REQUIRED_A1_VARIABLES = [
    "Sensor_Zenith",
]

REQUIRED_A2_VARIABLES = [
    "DNB_BRDF_Corrected_NTL",
    "Gap_Filled_DNB_BRDF_Corrected_NTL",
    "Mandatory_Quality_Flag",
]

missing_a1 = [
    variable
    for variable in REQUIRED_A1_VARIABLES
    if variable not in vnp46a1
]

missing_a2 = [
    variable
    for variable in REQUIRED_A2_VARIABLES
    if variable not in vnp46a2
]

if missing_a1:
    raise KeyError(
        "Missing VNP46A1 variables: "
        f"{missing_a1}\n"
        f"Available variables: {list(vnp46a1.data_vars)}"
    )

if missing_a2:
    raise KeyError(
        "Missing VNP46A2 variables: "
        f"{missing_a2}\n"
        f"Available variables: {list(vnp46a2.data_vars)}"
    )

for coordinate in ["x", "y"]:
    if coordinate not in vnp46a1.coords:
        raise KeyError(
            f"VNP46A1 is missing `{coordinate}`."
        )

    if coordinate not in vnp46a2.coords:
        raise KeyError(
            f"VNP46A2 is missing `{coordinate}`."
        )

if not np.allclose(
    vnp46a1["x"].values,
    vnp46a2["x"].values,
):
    raise ValueError(
        "VNP46A1 and VNP46A2 x coordinates do not match."
    )

if not np.allclose(
    vnp46a1["y"].values,
    vnp46a2["y"].values,
):
    raise ValueError(
        "VNP46A1 and VNP46A2 y coordinates do not match."
    )

full_dates = pd.date_range(
    ANALYSIS_START,
    ANALYSIS_END,
    freq="D",
)

vnp46a1 = (
    vnp46a1
    .sel(
        date=slice(
            ANALYSIS_START,
            ANALYSIS_END,
        )
    )
    .reindex(
        date=full_dates
    )
)

vnp46a2 = (
    vnp46a2
    .sel(
        date=slice(
            ANALYSIS_START,
            ANALYSIS_END,
        )
    )
    .reindex(
        date=full_dates
    )
)

print("Expected calendar days:", len(full_dates))
print(
    "VNP46A1 dates after reindexing:",
    vnp46a1.sizes["date"],
)
print(
    "VNP46A2 dates after reindexing:",
    vnp46a2.sizes["date"],
)

In [ ]:
# %% Cell 5 — load Notebook 00 quality-control outputs

pilot_qc_path = PROCESSED_DIR / "pilot_qc.csv"
processing_log_path = PROCESSED_DIR / "processing_log.csv"
source_manifest_path = PROCESSED_DIR / "source_manifest.csv"
pilot_qc_passed_path = PROCESSED_DIR / "pilot_qc_passed.json"

pilot_qc = (
    pd.read_csv(pilot_qc_path)
    if pilot_qc_path.exists()
    else pd.DataFrame()
)

processing_log = (
    pd.read_csv(processing_log_path)
    if processing_log_path.exists()
    else pd.DataFrame()
)

source_manifest = (
    pd.read_csv(source_manifest_path)
    if source_manifest_path.exists()
    else pd.DataFrame()
)

if pilot_qc_passed_path.exists():
    with open(
        pilot_qc_passed_path,
        "r",
    ) as file:
        pilot_qc_passed = json.load(file)
else:
    pilot_qc_passed = None

print("Pilot QC result:", pilot_qc_passed)
print("Pilot QC records:", len(pilot_qc))
print("Processing-log records:", len(processing_log))
print("Source-manifest records:", len(source_manifest))



In [ ]:
# %% Cell 6 — clean the required bands

def clean_band(data_array):
    cleaned = data_array.squeeze(
        drop=True
    ).astype("float32")

    for attribute in [
        "_FillValue",
        "missing_value",
        "nodata",
    ]:
        fill_value = data_array.attrs.get(
            attribute
        )

        if fill_value is None:
            continue

        fill_values = np.atleast_1d(
            fill_value
        )

        for value in fill_values:
            try:
                cleaned = cleaned.where(
                    cleaned != float(value)
                )
            except (TypeError, ValueError):
                pass

    return cleaned.where(
        np.isfinite(cleaned)
    )


dnb = clean_band(
    vnp46a2[
        "DNB_BRDF_Corrected_NTL"
    ]
).where(
    lambda values: (
        (values >= 0)
        & (values <= 6553.4)
    )
)

gap_filled = clean_band(
    vnp46a2[
        "Gap_Filled_DNB_BRDF_Corrected_NTL"
    ]
).where(
    lambda values: (
        (values >= 0)
        & (values <= 6553.4)
    )
)

mqf = clean_band(
    vnp46a2[
        "Mandatory_Quality_Flag"
    ]
)

sensor_zenith = clean_band(
    vnp46a1[
        "Sensor_Zenith"
    ]
)

# Handle an unscaled native VNP46A1 angle only if Notebook 00
# retained the original 0–9000 integer representation.
vza_sample = (
    sensor_zenith
    .isel(
        date=slice(
            0,
            min(
                30,
                sensor_zenith.sizes["date"],
            ),
        )
    )
    .max(
        skipna=True
    )
    .compute()
    .item()
)

if np.isfinite(vza_sample) and vza_sample > 180:
    sensor_zenith = (
        sensor_zenith
        * 0.01
    )

sensor_zenith = sensor_zenith.where(
    (sensor_zenith >= 0)
    & (sensor_zenith <= 90)
)

print(
    "DNB dimensions:",
    dict(dnb.sizes),
)

print(
    "Sensor-zenith range:",
    float(
        sensor_zenith.min(
            skipna=True
        ).compute()
    ),
    "to",
    float(
        sensor_zenith.max(
            skipna=True
        ).compute()
    ),
)


In [ ]:
# %% Cell 7 — align GHSL to the processed VNP46 grid

def get_dataset_crs(dataset):
    if "spatial_ref" in dataset.variables:
        attributes = dataset[
            "spatial_ref"
        ].attrs

        for key in [
            "crs_wkt",
            "spatial_ref",
        ]:
            value = attributes.get(key)

            if value:
                return CRS.from_user_input(
                    value
                )

        epsg_code = attributes.get(
            "epsg_code"
        )

        if epsg_code:
            return CRS.from_user_input(
                epsg_code
            )

    for key in [
        "crs",
        "crs_wkt",
    ]:
        value = dataset.attrs.get(key)

        if value:
            return CRS.from_user_input(
                value
            )

    x_values = dataset["x"].values
    y_values = dataset["y"].values

    if (
        np.nanmin(x_values) >= -180
        and np.nanmax(x_values) <= 180
        and np.nanmin(y_values) >= -90
        and np.nanmax(y_values) <= 90
    ):
        return CRS.from_epsg(4326)

    raise ValueError(
        "The VNP46 CRS is missing from the processed Zarr metadata."
    )


def get_dataset_transform(dataset):
    x_values = np.asarray(
        dataset["x"].values,
        dtype=float,
    )

    y_values = np.asarray(
        dataset["y"].values,
        dtype=float,
    )

    if len(x_values) < 2 or len(y_values) < 2:
        raise ValueError(
            "At least two x and y coordinates are required."
        )

    x_resolution = float(
        np.median(
            np.diff(x_values)
        )
    )

    y_resolution = float(
        np.median(
            np.diff(y_values)
        )
    )

    return Affine(
        x_resolution,
        0,
        x_values[0] - x_resolution / 2,
        0,
        y_resolution,
        y_values[0] - y_resolution / 2,
    )


reference_crs = get_dataset_crs(
    vnp46a2
)

reference_transform = get_dataset_transform(
    vnp46a2
)

reference_height = vnp46a2.sizes["y"]
reference_width = vnp46a2.sizes["x"]

ghsl_aligned = np.full(
    (
        reference_height,
        reference_width,
    ),
    np.nan,
    dtype="float32",
)

with rasterio.open(GHSL_PATH) as source:
    reproject(
        source=rasterio.band(
            source,
            1,
        ),
        destination=ghsl_aligned,
        src_transform=source.transform,
        src_crs=source.crs,
        src_nodata=source.nodata,
        dst_transform=reference_transform,
        dst_crs=reference_crs,
        dst_nodata=np.nan,
        resampling=Resampling.nearest,
    )

ghsl = xr.DataArray(
    ghsl_aligned,
    dims=(
        "y",
        "x",
    ),
    coords={
        "y": vnp46a2["y"],
        "x": vnp46a2["x"],
    },
    name="GHSL_SMOD_E2015",
)

ghsl_mask = xr.DataArray(
    np.isin(
        ghsl_aligned,
        GHSL_CLASSES,
    ),
    dims=(
        "y",
        "x",
    ),
    coords={
        "y": vnp46a2["y"],
        "x": vnp46a2["x"],
    },
    name="ghsl_g7_mask",
)

ghsl_pixel_count = int(
    ghsl_mask.sum().item()
)

if ghsl_pixel_count == 0:
    available_classes = np.unique(
        ghsl_aligned[
            np.isfinite(
                ghsl_aligned
            )
        ]
    )

    raise ValueError(
        "GHSL was aligned successfully, but no G7 pixels "
        f"were found. Available aligned classes: {available_classes}"
    )

print("VNP46 CRS:", reference_crs)
print("Aligned GHSL classes:", np.unique(
    ghsl_aligned[
        np.isfinite(
            ghsl_aligned
        )
    ]
))
print("GHSL G7 pixels:", f"{ghsl_pixel_count:,}")



In [ ]:
# %% Cell 8 — define the fixed pre-Haiyan settlement-light mask

fresh_dnb_ghsl = dnb.where(
    (mqf == 0)
    & ghsl_mask
)

baseline_fresh = fresh_dnb_ghsl.sel(
    date=slice(
        ANALYSIS_START,
        EVENT_DATE - pd.Timedelta(days=1),
    )
)

baseline_observed_days = (
    baseline_fresh
    .notnull()
    .sum(
        dim="date"
    )
    .compute()
)

baseline_median = (
    baseline_fresh
    .median(
        dim="date",
        skipna=True,
    )
    .compute()
)

fixed_mask = (
    ghsl_mask
    & (
        baseline_observed_days
        >= BASELINE_MIN_OBSERVED_DAYS
    )
    & (
        baseline_median
        > BASELINE_MIN_RADIANCE
    )
)

fixed_pixel_count = int(
    fixed_mask.sum().item()
)

if fixed_pixel_count == 0:
    raise ValueError(
        "The fixed signal mask contains no pixels. "
        "Inspect GHSL_CLASSES, BASELINE_MIN_OBSERVED_DAYS, "
        "and BASELINE_MIN_RADIANCE."
    )

fresh_dnb = dnb.where(
    (mqf == 0)
    & fixed_mask
)

gap_filled_fixed = gap_filled.where(
    fixed_mask
)

paired_gap_filled = gap_filled.where(
    fresh_dnb.notnull()
)

vza_observed = sensor_zenith.where(
    fresh_dnb.notnull()
)

baseline_reference = baseline_median.where(
    fixed_mask
    & (
        baseline_median > 0
    )
)

baseline_relative_pixel_anomaly = (
    fresh_dnb
    / baseline_reference
    - 1
)

print(
    "Baseline calendar days:",
    baseline_fresh.sizes["date"],
)
print(
    "GHSL G7 pixels:",
    f"{ghsl_pixel_count:,}",
)
print(
    "Fixed signal-mask pixels:",
    f"{fixed_pixel_count:,}",
)
print(
    "Retained from GHSL G7:",
    f"{100 * fixed_pixel_count / ghsl_pixel_count:.1f}%",
)


In [ ]:
# %% Cell 9 — Daily fixed-mask signal, observability, and VZA summaries

spatial_dimensions = [
    "y",
    "x",
]

# Count each MQF state within the fixed settlement-light mask.
mqf_0_count = (
    (
        fixed_mask
        & (mqf == 0)
    )
    .sum(
        dim=spatial_dimensions,
    )
)

mqf_1_count = (
    (
        fixed_mask
        & (mqf == 1)
    )
    .sum(
        dim=spatial_dimensions,
    )
)

mqf_2_count = (
    (
        fixed_mask
        & (mqf >= 2)
    )
    .sum(
        dim=spatial_dimensions,
    )
)

# Pixels not assigned to any valid MQF category are treated as
# having no retrieval. This also keeps the four percentages summing
# to approximately 100% on every date.
no_retrieval_count = (
    fixed_pixel_count
    - mqf_0_count
    - mqf_1_count
    - mqf_2_count
).clip(
    min=0,
)

# Calculate both VZA quantiles together, then remove the quantile
# coordinate from each selected result before constructing the Dataset.
# This prevents conflicting scalar coordinates such as quantile=0.10
# and quantile=0.90 from raising an xarray MergeError.
vza_quantiles = (
    vza_observed
    .quantile(
        [
            0.10,
            0.90,
        ],
        dim=spatial_dimensions,
        skipna=True,
    )
)

vza_p10 = vza_quantiles.sel(
    quantile=0.10,
    drop=True,
)

vza_p90 = vza_quantiles.sel(
    quantile=0.90,
    drop=True,
)

daily_reductions = xr.Dataset(
    {
        "fresh_dnb_mean": fresh_dnb.mean(
            dim=spatial_dimensions,
            skipna=True,
        ),
        "fresh_dnb_median": fresh_dnb.median(
            dim=spatial_dimensions,
            skipna=True,
        ),
        "gap_filled_full_mean": gap_filled_fixed.mean(
            dim=spatial_dimensions,
            skipna=True,
        ),
        "gap_filled_paired_mean": paired_gap_filled.mean(
            dim=spatial_dimensions,
            skipna=True,
        ),
        "baseline_relative_anomaly": (
            baseline_relative_pixel_anomaly
            .median(
                dim=spatial_dimensions,
                skipna=True,
            )
        ),
        "vza_median": vza_observed.median(
            dim=spatial_dimensions,
            skipna=True,
        ),
        "vza_p10": vza_p10,
        "vza_p90": vza_p90,
        "fresh_dnb_mean_vza_le_40": (
            fresh_dnb
            .where(
                sensor_zenith <= 40
            )
            .mean(
                dim=spatial_dimensions,
                skipna=True,
            )
        ),
        "fresh_dnb_mean_vza_le_30": (
            fresh_dnb
            .where(
                sensor_zenith <= 30
            )
            .mean(
                dim=spatial_dimensions,
                skipna=True,
            )
        ),
        "mqf_0_count": mqf_0_count,
        "mqf_1_count": mqf_1_count,
        "mqf_2_count": mqf_2_count,
        "no_retrieval_count": no_retrieval_count,
    }
).compute()

daily_summary = pd.DataFrame(
    {
        "date": pd.DatetimeIndex(
            daily_reductions["date"].values
        ),
        "fresh_dnb_mean": (
            daily_reductions[
                "fresh_dnb_mean"
            ].values
        ),
        "fresh_dnb_median": (
            daily_reductions[
                "fresh_dnb_median"
            ].values
        ),
        "gap_filled_full_mean": (
            daily_reductions[
                "gap_filled_full_mean"
            ].values
        ),
        "gap_filled_paired_mean": (
            daily_reductions[
                "gap_filled_paired_mean"
            ].values
        ),
        "baseline_relative_anomaly": (
            daily_reductions[
                "baseline_relative_anomaly"
            ].values
        ),
        "vza_median": (
            daily_reductions[
                "vza_median"
            ].values
        ),
        "vza_p10": (
            daily_reductions[
                "vza_p10"
            ].values
        ),
        "vza_p90": (
            daily_reductions[
                "vza_p90"
            ].values
        ),
        "fresh_dnb_mean_vza_le_40": (
            daily_reductions[
                "fresh_dnb_mean_vza_le_40"
            ].values
        ),
        "fresh_dnb_mean_vza_le_30": (
            daily_reductions[
                "fresh_dnb_mean_vza_le_30"
            ].values
        ),
        "mqf_0_count": (
            daily_reductions[
                "mqf_0_count"
            ].values
        ),
        "mqf_1_count": (
            daily_reductions[
                "mqf_1_count"
            ].values
        ),
        "mqf_2_count": (
            daily_reductions[
                "mqf_2_count"
            ].values
        ),
        "no_retrieval_count": (
            daily_reductions[
                "no_retrieval_count"
            ].values
        ),
    }
)

daily_summary = (
    daily_summary
    .sort_values("date")
    .reset_index(drop=True)
)

daily_summary["relative_day"] = (
    daily_summary["date"]
    - EVENT_DATE
).dt.days

daily_summary["mqf_0_pct"] = (
    100
    * daily_summary["mqf_0_count"]
    / fixed_pixel_count
)

daily_summary["mqf_1_pct"] = (
    100
    * daily_summary["mqf_1_count"]
    / fixed_pixel_count
)

daily_summary["mqf_2_pct"] = (
    100
    * daily_summary["mqf_2_count"]
    / fixed_pixel_count
)

daily_summary["no_retrieval_pct"] = (
    100
    * daily_summary["no_retrieval_count"]
    / fixed_pixel_count
)

daily_summary["qualified_90pct"] = (
    daily_summary["mqf_0_pct"]
    >= PHASE_COMPLETENESS_THRESHOLD
)

daily_summary["operational_divergence"] = (
    daily_summary["gap_filled_full_mean"]
    - daily_summary["fresh_dnb_mean"]
)

daily_summary["paired_divergence"] = (
    daily_summary["gap_filled_paired_mean"]
    - daily_summary["fresh_dnb_mean"]
)

daily_summary["phase"] = pd.NA

for definition in PHASE_WINDOWS:
    phase_mask = daily_summary[
        "relative_day"
    ].between(
        definition["start_day"],
        definition["end_day"],
        inclusive="both",
    )

    daily_summary.loc[
        phase_mask,
        "phase",
    ] = definition["phase"]

daily_summary["phase"] = pd.Categorical(
    daily_summary["phase"],
    categories=PHASE_ORDER,
    ordered=True,
)

daily_summary.to_csv(
    TABLE_DIR / "daily_fixed_mask_summary.csv",
    index=False,
)

display(
    daily_summary.head().style.format(
        {
            "fresh_dnb_mean": "{:.3f}",
            "fresh_dnb_median": "{:.3f}",
            "gap_filled_full_mean": "{:.3f}",
            "gap_filled_paired_mean": "{:.3f}",
            "operational_divergence": "{:.3f}",
            "paired_divergence": "{:.3f}",
            "baseline_relative_anomaly": "{:.3f}",
            "vza_median": "{:.2f}",
            "vza_p10": "{:.2f}",
            "vza_p90": "{:.2f}",
            "fresh_dnb_mean_vza_le_40": "{:.3f}",
            "fresh_dnb_mean_vza_le_30": "{:.3f}",
            "mqf_0_pct": "{:.1f}",
            "mqf_1_pct": "{:.1f}",
            "mqf_2_pct": "{:.1f}",
            "no_retrieval_pct": "{:.1f}",
        },
        na_rep="—",
    )
)

In [ ]:
# %% Cell 10 — shared Plotly helpers

def add_phase_shading(figure, rows):
    for definition in PHASE_WINDOWS:
        start_date = (
            EVENT_DATE
            + pd.Timedelta(
                days=definition["start_day"]
            )
        )

        end_date = (
            EVENT_DATE
            + pd.Timedelta(
                days=definition["end_day"] + 1
            )
        )

        for row in rows:
            figure.add_vrect(
                x0=start_date,
                x1=end_date,
                fillcolor=PHASE_FILL_COLORS[
                    definition["phase"]
                ],
                line_width=0,
                layer="below",
                row=row,
                col=1,
            )


def add_haiyan_marker(figure, rows):
    for row in rows:
        figure.add_vline(
            x=EVENT_DATE,
            line={
                "color": HAIYAN_COLOR,
                "width": 2,
                "dash": "dash",
            },
            row=row,
            col=1,
        )

    figure.add_annotation(
        x=EVENT_DATE,
        y=1.02,
        xref="x",
        yref="paper",
        text="Haiyan/Yolanda<br>8 Nov 2013",
        showarrow=False,
        xanchor="left",
        yanchor="bottom",
        font={
            "size": 13,
            "color": HAIYAN_COLOR,
        },
        bgcolor="rgba(255,255,255,0.85)",
    )


In [ ]:
# %% Cell 11 — Figure 1: GHSL-constrained fixed signal mask

import geopandas as gpd
from shapely.geometry import box


REGION_SHAPEFILE = Path(
    "./datasets/boundaries/regions/Regions.shp"
)

region_boundaries = (
    gpd.read_file(REGION_SHAPEFILE)
    .to_crs("EPSG:4326")
)

raster_x = vnp46a2["x"].values
raster_y = vnp46a2["y"].values

raster_extent = box(
    float(np.nanmin(raster_x)),
    float(np.nanmin(raster_y)),
    float(np.nanmax(raster_x)),
    float(np.nanmax(raster_y)),
)

plot_geometry = (
    region_boundaries.loc[
        region_boundaries.geometry.intersects(raster_extent),
        "geometry",
    ]
    .union_all()
    .intersection(raster_extent)
)


def polygon_lines(geometry):
    lines = []

    if geometry.is_empty:
        return lines

    if geometry.geom_type == "Polygon":
        polygons = [geometry]
    elif geometry.geom_type == "MultiPolygon":
        polygons = list(geometry.geoms)
    else:
        polygons = [
            item
            for item in geometry.geoms
            if item.geom_type == "Polygon"
        ]

    for polygon in polygons:
        x, y = polygon.exterior.xy
        lines.append(
            (list(x), list(y))
        )

    return lines


boundary_lines = polygon_lines(
    plot_geometry
)

baseline_values = (
    baseline_median
    .where(fixed_mask)
    .values
)

finite_values = baseline_values[
    np.isfinite(baseline_values)
]

if len(finite_values) == 0:
    raise ValueError(
        "No finite baseline radiance remains in the fixed mask."
    )

radiance_color_max = max(
    float(np.nanpercentile(finite_values, 98)),
    1.0,
)

regional_x_range = [
    float(np.nanmin(raster_x)),
    float(np.nanmax(raster_x)),
]

regional_y_range = [
    float(np.nanmin(raster_y)),
    float(np.nanmax(raster_y)),
]

tacloban_x_range = [
    124.94,
    125.10,
]

tacloban_y_range = [
    11.14,
    11.32,
]


def add_boundary(fig, row=None, col=None, xaxis=None, yaxis=None):
    for x, y in boundary_lines:
        trace = go.Scatter(
            x=x,
            y=y,
            mode="lines",
            line={
                "color": "#64748B",
                "width": 0.8,
            },
            hoverinfo="skip",
            showlegend=False,
        )

        if xaxis is not None:
            trace.update(
                xaxis=xaxis,
                yaxis=yaxis,
            )
            fig.add_trace(trace)
        else:
            fig.add_trace(
                trace,
                row=row,
                col=col,
            )


def add_map(fig, z, row, col, colorscale, showscale=False, colorbar=None):
    fig.add_trace(
        go.Heatmap(
            x=raster_x,
            y=raster_y,
            z=z,
            zmin=0,
            zmax=(
                radiance_color_max
                if showscale
                else 1
            ),
            colorscale=colorscale,
            showscale=showscale,
            colorbar=colorbar,
            hovertemplate=(
                "Longitude: %{x:.4f}°<br>"
                "Latitude: %{y:.4f}°<br>"
                "Value: %{z}<extra></extra>"
            ),
        ),
        row=row,
        col=col,
    )


fig = make_subplots(
    rows=1,
    cols=3,
    horizontal_spacing=0.055,
    subplot_titles=[
        "<b>a.</b> GHSL G7 settlement pixels<br>classes 23 and 30",
        "<b>b.</b> Pre-Haiyan median DNB-BRDF<br>within GHSL G7",
        f"<b>c.</b> Fixed settlement-light mask<br>{fixed_pixel_count:,} retained pixels",
    ],
)

add_map(
    fig,
    np.where(ghsl_mask.values, 1.0, np.nan),
    1,
    1,
    [[0, "#DCFCE7"], [1, "#166534"]],
)

add_map(
    fig,
    baseline_median.where(ghsl_mask).values,
    1,
    2,
    "Inferno",
    showscale=True,
    colorbar={
        "title": {
            "text": "Median radiance<br>(nW cm⁻² sr⁻¹)",
        },
        "len": 0.62,
        "thickness": 15,
        "x": 0.655,
        "y": 0.45,
    },
)

add_map(
    fig,
    np.where(fixed_mask.values, 1.0, np.nan),
    1,
    3,
    [[0, "#DBEAFE"], [1, "#1D4ED8"]],
)

for column in [1, 2, 3]:
    add_boundary(
        fig,
        row=1,
        col=column,
    )

    fig.add_shape(
        type="rect",
        x0=tacloban_x_range[0],
        x1=tacloban_x_range[1],
        y0=tacloban_y_range[0],
        y1=tacloban_y_range[1],
        xref=(
            "x"
            if column == 1
            else f"x{column}"
        ),
        yref=(
            "y"
            if column == 1
            else f"y{column}"
        ),
        line={
            "color": "#DC2626",
            "width": 1.4,
            "dash": "dash",
        },
        fillcolor="rgba(0,0,0,0)",
    )

main_domains = [
    fig.layout.xaxis.domain,
    fig.layout.xaxis2.domain,
    fig.layout.xaxis3.domain,
]

main_y_domain = fig.layout.yaxis.domain
inset_domains = []

for domain in main_domains:
    inset_width = (
        domain[1] - domain[0]
    ) * 0.36

    inset_height = (
        main_y_domain[1] - main_y_domain[0]
    ) * 0.32

    inset_domains.append(
        {
            "x": [
                domain[1] - inset_width - 0.012,
                domain[1] - 0.012,
            ],
            "y": [
                main_y_domain[1] - inset_height - 0.025,
                main_y_domain[1] - 0.025,
            ],
        }
    )

inset_layers = [
    np.where(ghsl_mask.values, 1.0, np.nan),
    baseline_median.where(ghsl_mask).values,
    np.where(fixed_mask.values, 1.0, np.nan),
]

inset_scales = [
    [[0, "#DCFCE7"], [1, "#166534"]],
    "Inferno",
    [[0, "#DBEAFE"], [1, "#1D4ED8"]],
]

for idx, domain in enumerate(inset_domains, start=4):
    fig.update_layout(
        {
            f"xaxis{idx}": {
                "domain": domain["x"],
                "anchor": f"y{idx}",
                "range": tacloban_x_range,
                "showgrid": False,
                "zeroline": False,
                "showticklabels": False,
                "ticks": "",
            },
            f"yaxis{idx}": {
                "domain": domain["y"],
                "anchor": f"x{idx}",
                "range": tacloban_y_range,
                "showgrid": False,
                "zeroline": False,
                "showticklabels": False,
                "ticks": "",
                "scaleanchor": f"x{idx}",
                "scaleratio": 1,
            },
        }
    )

    fig.add_shape(
        type="rect",
        x0=domain["x"][0],
        x1=domain["x"][1],
        y0=domain["y"][0],
        y1=domain["y"][1],
        xref="paper",
        yref="paper",
        fillcolor="white",
        line={
            "color": "#475569",
            "width": 1.1,
        },
        layer="below",
    )

    fig.add_trace(
        go.Heatmap(
            x=raster_x,
            y=raster_y,
            z=inset_layers[idx - 4],
            xaxis=f"x{idx}",
            yaxis=f"y{idx}",
            zmin=0,
            zmax=(
                radiance_color_max
                if idx == 5
                else 1
            ),
            colorscale=inset_scales[idx - 4],
            showscale=False,
            hoverinfo="skip",
        )
    )

    add_boundary(
        fig,
        xaxis=f"x{idx}",
        yaxis=f"y{idx}",
    )

    fig.add_annotation(
        x=domain["x"][0] + 0.004,
        y=domain["y"][1] - 0.006,
        xref="paper",
        yref="paper",
        text="<b>Tacloban</b>",
        showarrow=False,
        xanchor="left",
        yanchor="top",
        font={
            "size": 11,
            "color": "#111827",
        },
        bgcolor="rgba(255,255,255,0.85)",
        borderpad=2,
    )

for column in [1, 2, 3]:
    x_reference = (
        "x"
        if column == 1
        else f"x{column}"
    )

    fig.update_xaxes(
        range=regional_x_range,
        showgrid=False,
        zeroline=False,
        showticklabels=False,
        ticks="",
        constrain="domain",
        row=1,
        col=column,
    )

    fig.update_yaxes(
        range=regional_y_range,
        showgrid=False,
        zeroline=False,
        showticklabels=False,
        ticks="",
        scaleanchor=x_reference,
        scaleratio=1,
        row=1,
        col=column,
    )

fig.update_layout(
    title={
        "text": "Fixed pre-Haiyan settlement-light mask within GHSL G7",
        "x": 0.01,
        "xanchor": "left",
    },
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    font={
        "family": "Arial",
        "size": 16,
        "color": "#111827",
    },
    margin={
        "l": 25,
        "r": 90,
        "t": 125,
        "b": 30,
    },
    width=1550,
    height=630,
    showlegend=False,
)

fig.show()

fig.write_html(
    FIGURE_DIR / "01_fixed_signal_mask.html",
    include_plotlyjs="cdn",
)

## Figure 1. Fixed Pre-Haiyan Settlement-Light Mask

Figure 1 defines the spatial support for all subsequent diagnostics. The mask is restricted to GHSL G7 settlement classes and further limited to pixels with sufficient pre-Haiyan direct DNB-BRDF observations and positive baseline radiance. This produces a fixed settlement-light mask of 2,585 pixels.

This step prevents the recovery analysis from being driven by unstable rural, dim, water-adjacent, or non-settlement pixels. It also ensures that pre-event lighting exists before any post-event deficit is interpreted as disaster impact.

The mask should be interpreted as an admissible urban-light sample, not as the full population of affected settlements. Later RQ2 extensions can repeat the same logic across other GHSL classes and 1x1, 3x3, and 5x5 local extents.


In [ ]:
subplot_titles = [
    (
        f"<b>{record['phase']}</b><br>"
        f"≥90% dates: {record['qualified_dates']} / {record['total_days']}"
    )
    for record in phase_composite_records
]

fig = make_subplots(
    rows=1,
    cols=len(PHASE_ORDER),
    horizontal_spacing=0.03,
    subplot_titles=subplot_titles,
)

for column, phase_name in enumerate(PHASE_ORDER, start=1):
    fig.add_trace(
        go.Heatmap(
            x=raster_x,
            y=raster_y,
            z=phase_composites[phase_name].values,
            coloraxis="coloraxis",
            hovertemplate=(
                "Longitude: %{x:.4f}°<br>"
                "Latitude: %{y:.4f}°<br>"
                "Median DNB-BRDF: %{z:.3f}"
                "<extra></extra>"
            ),
        ),
        row=1,
        col=column,
    )

    add_boundary(
        fig,
        row=1,
        col=column,
    )

    fig.add_shape(
        type="rect",
        x0=tacloban_x_range[0],
        x1=tacloban_x_range[1],
        y0=tacloban_y_range[0],
        y1=tacloban_y_range[1],
        xref=("x" if column == 1 else f"x{column}"),
        yref=("y" if column == 1 else f"y{column}"),
        line={
            "color": "#DC2626",
            "width": 1.3,
            "dash": "dash",
        },
        fillcolor="rgba(0,0,0,0)",
    )

main_domains = [
    fig.layout[f"xaxis{'' if i == 1 else i}"].domain
    for i in range(1, len(PHASE_ORDER) + 1)
]

main_y_domain = fig.layout.yaxis.domain
inset_domains = []

for domain in main_domains:
    inset_width = (domain[1] - domain[0]) * 0.42
    inset_height = (main_y_domain[1] - main_y_domain[0]) * 0.34

    inset_domains.append(
        {
            "x": [
                domain[1] - inset_width - 0.008,
                domain[1] - 0.008,
            ],
            "y": [
                main_y_domain[1] - inset_height - 0.025,
                main_y_domain[1] - 0.025,
            ],
        }
    )

for idx, (domain, phase_name) in enumerate(
    zip(inset_domains, PHASE_ORDER),
    start=len(PHASE_ORDER) + 1,
):
    fig.update_layout(
        {
            f"xaxis{idx}": {
                "domain": domain["x"],
                "anchor": f"y{idx}",
                "range": tacloban_x_range,
                "showgrid": False,
                "zeroline": False,
                "showticklabels": False,
                "ticks": "",
            },
            f"yaxis{idx}": {
                "domain": domain["y"],
                "anchor": f"x{idx}",
                "range": tacloban_y_range,
                "showgrid": False,
                "zeroline": False,
                "showticklabels": False,
                "ticks": "",
                "scaleanchor": f"x{idx}",
                "scaleratio": 1,
            },
        }
    )

    fig.add_shape(
        type="rect",
        x0=domain["x"][0],
        x1=domain["x"][1],
        y0=domain["y"][0],
        y1=domain["y"][1],
        xref="paper",
        yref="paper",
        fillcolor="white",
        line={
            "color": "#475569",
            "width": 1.1,
        },
        layer="below",
    )

    fig.add_trace(
        go.Heatmap(
            x=raster_x,
            y=raster_y,
            z=phase_composites[phase_name].values,
            xaxis=f"x{idx}",
            yaxis=f"y{idx}",
            coloraxis="coloraxis",
            showscale=False,
            hoverinfo="skip",
        )
    )

    add_boundary(
        fig,
        xaxis=f"x{idx}",
        yaxis=f"y{idx}",
    )

    fig.add_annotation(
        x=domain["x"][0] + 0.003,
        y=domain["y"][1] - 0.005,
        xref="paper",
        yref="paper",
        text="<b>Tacloban</b>",
        showarrow=False,
        xanchor="left",
        yanchor="top",
        font={
            "size": 10,
            "color": "#111827",
        },
        bgcolor="rgba(255,255,255,0.85)",
        borderpad=2,
    )

for column in range(1, len(PHASE_ORDER) + 1):
    x_reference = "x" if column == 1 else f"x{column}"

    fig.update_xaxes(
        range=regional_x_range,
        showticklabels=False,
        ticks="",
        showgrid=False,
        zeroline=False,
        constrain="domain",
        row=1,
        col=column,
    )

    fig.update_yaxes(
        range=regional_y_range,
        showticklabels=False,
        ticks="",
        showgrid=False,
        zeroline=False,
        scaleanchor=x_reference,
        scaleratio=1,
        row=1,
        col=column,
    )

fig.update_layout(
    title={
        "text": (
            "Direct DNB-BRDF phase composites restricted "
            "to dates with ≥90% MQF = 0 completeness"
        ),
        "x": 0.01,
        "xanchor": "left",
    },
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    font={
        "family": "Arial",
        "size": 16,
        "color": "#111827",
    },
    coloraxis={
        "colorscale": "Inferno",
        "cmin": 0,
        "cmax": radiance_color_max,
        "colorbar": {
            "title": {
                "text": "Median radiance<br>(nW cm⁻² sr⁻¹)",
            },
            "len": 0.72,
            "thickness": 16,
            "x": 1.01,
            "y": 0.48,
        },
    },
    margin={
        "l": 25,
        "r": 115,
        "t": 125,
        "b": 35,
    },
    width=1850,
    height=520,
    showlegend=False,
)

fig.show()

fig.write_html(
    FIGURE_DIR / "02_phase_90pct_dnb_composites.html",
    include_plotlyjs="cdn",
)

### Figure 2. Phase Composites Under High-Observability Conditions

Figure 2 compares median direct DNB-BRDF radiance across the baseline, shock, early recovery, late recovery, and long-term phases using only dates with at least 90% MQF = 0 completeness. This restriction makes the phase composites more defensible because each included date has broad fresh-observation support across the fixed mask.

The result should be read spatially rather than as a precise recovery timeline. The composites can show whether settlement-light patterns remain visibly suppressed or spatially reorganized after Haiyan, but they cannot identify exact recovery dates. Phases with fewer high-completeness dates should be interpreted as weaker evidence, even if their composite appears visually clear.

The main value of this figure is that it converts cloud-limited daily observations into a controlled phase comparison. It supports broad phase-level interpretation, not fine-grained T50 or T80 inference.


In [ ]:
# %% Cell 13 — Figure 3: direct versus gap-filled nighttime lights

marker_sizes = (
    4
    + 8
    * daily_summary["mqf_0_pct"]
    / 100
)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.10,
    row_heights=[
        0.67,
        0.33,
    ],
    subplot_titles=[
        (
            "Fixed-mask mean radiance; marker size "
            "represents MQF = 0 completeness"
        ),
        (
            "Gap-filled divergence relative to "
            "direct DNB-BRDF"
        ),
    ],
)

fig.add_trace(
    go.Scatter(
        x=daily_summary["date"],
        y=daily_summary[
            "gap_filled_full_mean"
        ],
        mode="lines",
        name="Gap-filled: full fixed mask",
        line={
            "color": "#EA580C",
            "width": 2.2,
            "dash": "dot",
        },
        connectgaps=False,
        customdata=daily_summary[
            "mqf_0_pct"
        ],
        hovertemplate=(
            "%{x|%Y-%m-%d}<br>"
            "Gap-filled mean: %{y:.3f}<br>"
            "MQF = 0: %{customdata:.1f}%"
            "<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=daily_summary["date"],
        y=daily_summary[
            "fresh_dnb_mean"
        ],
        mode="lines+markers",
        name="Direct DNB-BRDF: MQF = 0",
        line={
            "color": "#111827",
            "width": 2,
        },
        marker={
            "size": marker_sizes,
            "color": daily_summary[
                "mqf_0_pct"
            ],
            "colorscale": [
                [0, "#CBD5E1"],
                [1, "#16A34A"],
            ],
            "cmin": 0,
            "cmax": 100,
            "showscale": False,
            "line": {
                "color": "white",
                "width": 0.4,
            },
        },
        connectgaps=False,
        customdata=np.column_stack(
            [
                daily_summary[
                    "mqf_0_pct"
                ],
                daily_summary[
                    "relative_day"
                ],
            ]
        ),
        hovertemplate=(
            "%{x|%Y-%m-%d}<br>"
            "Relative day: %{customdata[1]:+.0f}<br>"
            "Direct mean: %{y:.3f}<br>"
            "MQF = 0: %{customdata[0]:.1f}%"
            "<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=daily_summary["date"],
        y=daily_summary[
            "operational_divergence"
        ],
        mode="lines",
        name=(
            "Full-mask gap-filled − "
            "direct observed"
        ),
        line={
            "color": "#DC2626",
            "width": 2,
        },
        connectgaps=False,
        customdata=daily_summary[
            "mqf_0_pct"
        ],
        hovertemplate=(
            "%{x|%Y-%m-%d}<br>"
            "Operational divergence: %{y:.3f}<br>"
            "MQF = 0: %{customdata:.1f}%"
            "<extra></extra>"
        ),
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=daily_summary["date"],
        y=daily_summary[
            "paired_divergence"
        ],
        mode="lines",
        name=(
            "Paired-pixel gap-filled − "
            "direct observed"
        ),
        line={
            "color": "#7C3AED",
            "width": 1.7,
            "dash": "dot",
        },
        connectgaps=False,
        customdata=daily_summary[
            "mqf_0_pct"
        ],
        hovertemplate=(
            "%{x|%Y-%m-%d}<br>"
            "Paired divergence: %{y:.3f}<br>"
            "MQF = 0: %{customdata:.1f}%"
            "<extra></extra>"
        ),
    ),
    row=2,
    col=1,
)

fig.add_hline(
    y=0,
    line={
        "color": "#64748B",
        "width": 1.2,
        "dash": "dash",
    },
    row=2,
    col=1,
)

add_phase_shading(
    fig,
    rows=[
        1,
        2,
    ],
)

add_haiyan_marker(
    fig,
    rows=[
        1,
        2,
    ],
)

fig.update_yaxes(
    title_text=(
        "Mean radiance<br>"
        "(nW cm⁻² sr⁻¹)"
    ),
    rangemode="tozero",
    row=1,
    col=1,
)

fig.update_yaxes(
    title_text=(
        "Difference<br>"
        "(nW cm⁻² sr⁻¹)"
    ),
    zeroline=False,
    row=2,
    col=1,
)

fig.update_xaxes(
    title_text="Date",
    row=2,
    col=1,
)

fig.update_layout(
    title={
        "text": (
            "Direct and gap-filled nighttime lights over "
            "the fixed pre-Haiyan settlement-light mask"
        ),
        "x": 0.01,
        "xanchor": "left",
    },
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    font={
        "family": "Arial",
        "size": 16,
        "color": "#111827",
    },
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.10,
        "xanchor": "right",
        "x": 1,
    },
    margin={
        "l": 95,
        "r": 40,
        "t": 145,
        "b": 70,
    },
    width=1450,
    height=850,
    hovermode="x unified",
)

fig.show()

fig.write_html(
    FIGURE_DIR / "03_observed_vs_gap_filled.html",
    include_plotlyjs="cdn",
)



### Figure 3. Direct Versus Gap-Filled Nighttime Lights

Figure 3 compares direct DNB-BRDF observations against gap-filled NTL over the same fixed settlement-light mask. The direct series is visibly jumpier because each date averages only the pixels that were freshly observed with MQF = 0. The contributing pixel set changes from day to day, so sharp variation may reflect changing observability and spatial sampling, not only real change in electrified activity.

The gap-filled series is smoother because it supplies values across the full mask. That smoothness should not be treated as stronger evidence of recovery. It may reduce noise, but it can also attenuate disaster shock, delay apparent impact, or create continuity during periods when direct observations are sparse.

The divergence panel is therefore central. When gap-filled NTL and direct DNB-BRDF disagree, the recovery story becomes product-sensitive. Those periods should be flagged before estimating shock magnitude, cumulative deficit, T50, or T80.



### Figure 3. Direct versus gap-filled nighttime lights

The direct DNB-BRDF series is visibly more irregular than the gap-filled series because it is based only on MQF = 0 observations. The contributing pixels vary by date, so short-term spikes and drops likely reflect changing observability and pixel composition as well as real lighting change.

The gap-filled series is smoother because it provides spatially complete values across the fixed mask, but this smoothness should not be interpreted as higher reliability. Divergence between gap-filled and direct observations indicates periods when the gap-filled product may attenuate, delay, or smooth the true observed signal.

This figure shows why recovery metrics should not be derived from raw daily direct NTL alone. Direct observations preserve the real observation constraint, while gap-filled NTL is useful as a diagnostic comparison for potential smoothing bias.


In [ ]:
# %% Cell 14 — Figure 4: observability through time and by phase

phase_observability = (
    daily_summary
    .groupby(
        "phase",
        observed=False,
    )
    .agg(
        mqf_0_pct=(
            "mqf_0_pct",
            "mean",
        ),
        mqf_1_pct=(
            "mqf_1_pct",
            "mean",
        ),
        mqf_2_pct=(
            "mqf_2_pct",
            "mean",
        ),
        no_retrieval_pct=(
            "no_retrieval_pct",
            "mean",
        ),
        mqf_0_median=(
            "mqf_0_pct",
            "median",
        ),
        mqf_0_q25=(
            "mqf_0_pct",
            lambda values: values.quantile(
                0.25
            ),
        ),
        mqf_0_q75=(
            "mqf_0_pct",
            lambda values: values.quantile(
                0.75
            ),
        ),
        days=(
            "date",
            "size",
        ),
        days_ge_90pct=(
            "qualified_90pct",
            "sum",
        ),
    )
    .reset_index()
)

phase_observability[
    "mqf_0_error_plus"
] = (
    phase_observability[
        "mqf_0_q75"
    ]
    - phase_observability[
        "mqf_0_median"
    ]
)

phase_observability[
    "mqf_0_error_minus"
] = (
    phase_observability[
        "mqf_0_median"
    ]
    - phase_observability[
        "mqf_0_q25"
    ]
)

phase_observability.to_csv(
    TABLE_DIR / "phase_observability_summary.csv",
    index=False,
)

daily_stack_definitions = [
    (
        "mqf_0_pct",
        "MQF = 0",
    ),
    (
        "mqf_1_pct",
        "MQF = 1",
    ),
    (
        "mqf_2_pct",
        "MQF ≥ 2",
    ),
    (
        "no_retrieval_pct",
        "No retrieval",
    ),
]

fig = make_subplots(
    rows=2,
    cols=1,
    vertical_spacing=0.15,
    row_heights=[
        0.62,
        0.38,
    ],
    subplot_titles=[
        "Daily composition of the fixed signal mask",
        (
            "Mean phase composition with median and IQR "
            "of daily MQF = 0 completeness"
        ),
    ],
)

for column, label in daily_stack_definitions:
    fig.add_trace(
        go.Bar(
            x=daily_summary["date"],
            y=daily_summary[column],
            name=label,
            marker={
                "color": MQF_COLORS[
                    label
                ],
                "line": {
                    "width": 0,
                },
            },
            hovertemplate=(
                "%{x|%Y-%m-%d}<br>"
                + label
                + ": %{y:.1f}%"
                "<extra></extra>"
            ),
        ),
        row=1,
        col=1,
    )

for column, label in daily_stack_definitions:
    fig.add_trace(
        go.Bar(
            x=phase_observability[
                "phase"
            ],
            y=phase_observability[
                column
            ],
            name=label,
            legendgroup=label,
            showlegend=False,
            marker={
                "color": MQF_COLORS[
                    label
                ],
                "line": {
                    "color": "white",
                    "width": 0.4,
                },
            },
            customdata=phase_observability[
                [
                    "days",
                    "days_ge_90pct",
                ]
            ],
            hovertemplate=(
                "%{x}<br>"
                + label
                + ": %{y:.1f}%<br>"
                "Phase days: %{customdata[0]}<br>"
                "≥90% dates: %{customdata[1]}"
                "<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

fig.add_trace(
    go.Scatter(
        x=phase_observability[
            "phase"
        ],
        y=phase_observability[
            "mqf_0_median"
        ],
        mode="markers",
        name="Median MQF = 0 and IQR",
        marker={
            "color": "#111827",
            "size": 11,
            "symbol": "diamond",
            "line": {
                "color": "white",
                "width": 1,
            },
        },
        error_y={
            "type": "data",
            "symmetric": False,
            "array": phase_observability[
                "mqf_0_error_plus"
            ],
            "arrayminus": phase_observability[
                "mqf_0_error_minus"
            ],
            "color": "#111827",
            "thickness": 2,
            "width": 6,
        },
        hovertemplate=(
            "%{x}<br>"
            "Median MQF = 0: %{y:.1f}%"
            "<extra></extra>"
        ),
    ),
    row=2,
    col=1,
)

fig.add_hline(
    y=PHASE_COMPLETENESS_THRESHOLD,
    line={
        "color": HAIYAN_COLOR,
        "width": 1.7,
        "dash": "dash",
    },
    row=1,
    col=1,
)

fig.add_hline(
    y=PHASE_COMPLETENESS_THRESHOLD,
    line={
        "color": HAIYAN_COLOR,
        "width": 1.7,
        "dash": "dash",
    },
    row=2,
    col=1,
)

add_phase_shading(
    fig,
    rows=[
        1,
    ],
)

add_haiyan_marker(
    fig,
    rows=[
        1,
    ],
)

fig.update_yaxes(
    title_text="Fixed-mask pixels (%)",
    range=[
        0,
        100,
    ],
    row=1,
    col=1,
)

fig.update_yaxes(
    title_text="Phase composition (%)",
    range=[
        0,
        100,
    ],
    row=2,
    col=1,
)

fig.update_xaxes(
    title_text="Phase",
    categoryorder="array",
    categoryarray=PHASE_ORDER,
    row=2,
    col=1,
)

fig.update_layout(
    title={
        "text": (
            "Direct observability across the Haiyan "
            "baseline, shock, recovery, and long-term phases"
        ),
        "x": 0.01,
        "xanchor": "left",
    },
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    font={
        "family": "Arial",
        "size": 16,
        "color": "#111827",
    },
    barmode="stack",
    bargap=0,
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.10,
        "xanchor": "right",
        "x": 1,
    },
    margin={
        "l": 95,
        "r": 40,
        "t": 145,
        "b": 80,
    },
    width=1450,
    height=900,
)

fig.show()

fig.write_html(
    FIGURE_DIR / "04_observability_daily_and_phase.html",
    include_plotlyjs="cdn",
)

display(
    phase_observability[
        [
            "phase",
            "days",
            "days_ge_90pct",
            "mqf_0_pct",
            "mqf_0_median",
            "mqf_0_q25",
            "mqf_0_q75",
            "mqf_1_pct",
            "mqf_2_pct",
            "no_retrieval_pct",
        ]
    ].style.format(
        {
            "mqf_0_pct": "{:.1f}",
            "mqf_0_median": "{:.1f}",
            "mqf_0_q25": "{:.1f}",
            "mqf_0_q75": "{:.1f}",
            "mqf_1_pct": "{:.1f}",
            "mqf_2_pct": "{:.1f}",
            "no_retrieval_pct": "{:.1f}",
        }
    )
)


### Figure 4. Daily and Phase-Level Observability

Figure 4 shows that observability varies strongly across the Haiyan analysis window. High-completeness dates exist in all phases, but the median MQF = 0 percentage remains uneven:

| Phase | Days | Dates with >=90% MQF = 0 | Mean MQF = 0 (%) | Median MQF = 0 (%) | Q25 (%) | Q75 (%) |
|---|---:|---:|---:|---:|---:|---:|
| Baseline | 180 | 33 | 19.2 | 1.6 | 0.0 | 29.7 |
| Shock | 31 | 15 | 39.8 | 37.8 | 6.7 | 65.7 |
| Early recovery | 60 | 17 | 25.7 | 8.1 | 0.0 | 50.6 |
| Late recovery | 90 | 43 | 47.6 | 44.5 | 13.3 | 84.0 |
| Long-term | 185 | 45 | 26.4 | 10.6 | 0.0 | 49.2 |

The shock and late recovery phases have stronger median observability than the baseline, early recovery, and long-term phases. However, all phases contain at least several high-completeness dates, which means broad phase-level comparison is possible under the current decision rule.

This does not mean daily recovery timing is reliable. The low median MQF = 0 values in several phases show that many dates remain observation-limited. The appropriate conclusion is that phase-level composites are admissible, while precise recovery metrics still require stricter filtering, smoothing, and uncertainty labels.


In [ ]:
# Required before Figure 5.

baseline_daily_signal = (
    daily_summary.loc[
        daily_summary["phase"] == "Baseline",
        "fresh_dnb_mean",
    ]
    .dropna()
)

baseline_daily_mean = baseline_daily_signal.mean()
baseline_daily_std = baseline_daily_signal.std()

if (
    not np.isfinite(baseline_daily_std)
    or baseline_daily_std == 0
):
    raise ValueError(
        "Baseline daily radiance has no usable variance."
    )

daily_summary["fresh_dnb_standardised"] = (
    daily_summary["fresh_dnb_mean"] - baseline_daily_mean
) / baseline_daily_std

In [ ]:
# Use smoothed lines for visual interpretation only.
daily_summary["vza_median_14d"] = (
    daily_summary["vza_median"]
    .rolling(
        window=14,
        center=True,
        min_periods=3,
    )
    .median()
)

daily_summary["fresh_dnb_standardised_14d"] = (
    daily_summary["fresh_dnb_standardised"]
    .rolling(
        window=14,
        center=True,
        min_periods=3,
    )
    .median()
)

fig = make_subplots(
    rows=1,
    cols=2,
    specs=[
        [
            {"secondary_y": True},
            {"secondary_y": False},
        ]
    ],
    subplot_titles=[
        "Smoothed VZA and standardised direct DNB-BRDF",
        "Baseline-relative radiance anomaly by VZA bin",
    ],
    horizontal_spacing=0.13,
)

# ------------------------------------------------------------
# Panel A: temporal VZA and radiance
# ------------------------------------------------------------

fig.add_trace(
    go.Scatter(
        x=daily_summary["date"],
        y=daily_summary["vza_p90"],
        mode="lines",
        line={"color": "rgba(124,58,237,0)"},
        hoverinfo="skip",
        showlegend=False,
    ),
    row=1,
    col=1,
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=daily_summary["date"],
        y=daily_summary["vza_p10"],
        mode="lines",
        fill="tonexty",
        fillcolor="rgba(124,58,237,0.13)",
        line={"color": "rgba(124,58,237,0)"},
        name="VZA P10-P90",
        hoverinfo="skip",
    ),
    row=1,
    col=1,
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=daily_summary["date"],
        y=daily_summary["vza_median_14d"],
        mode="lines",
        name="Median VZA, 14-day median",
        line={
            "color": "#7C3AED",
            "width": 2.5,
        },
        customdata=daily_summary["mqf_0_pct"],
        hovertemplate=(
            "%{x|%Y-%m-%d}<br>"
            "Median VZA: %{y:.2f}°<br>"
            "MQF = 0: %{customdata:.1f}%"
            "<extra></extra>"
        ),
    ),
    row=1,
    col=1,
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=daily_summary["date"],
        y=daily_summary["fresh_dnb_standardised_14d"],
        mode="lines",
        name="DNB-BRDF, 14-day median",
        line={
            "color": "#111827",
            "width": 2.5,
        },
        connectgaps=False,
        customdata=daily_summary["mqf_0_pct"],
        hovertemplate=(
            "%{x|%Y-%m-%d}<br>"
            "Standardised DNB-BRDF: %{y:.2f}<br>"
            "MQF = 0: %{customdata:.1f}%"
            "<extra></extra>"
        ),
    ),
    row=1,
    col=1,
    secondary_y=True,
)

# ------------------------------------------------------------
# Panel B: VZA-bin sensitivity
# ------------------------------------------------------------

for phase_name in ["Baseline", "Long-term"]:
    phase_data = vza_scatter_data.loc[
        vza_scatter_data["phase"] == phase_name
    ]

    fig.add_trace(
        go.Scatter(
            x=phase_data["vza_median"],
            y=100 * phase_data["baseline_relative_anomaly"],
            mode="markers",
            name=f"{phase_name} daily values",
            legendgroup=phase_name,
            showlegend=False,
            marker={
                "size": 5,
                "color": PHASE_COLORS[phase_name],
                "opacity": 0.20,
                "line": {
                    "color": "white",
                    "width": 0.3,
                },
            },
            customdata=phase_data["mqf_0_pct"].to_numpy(),
            hovertemplate=(
                "Median VZA: %{x:.2f}°<br>"
                "Anomaly: %{y:.1f}%<br>"
                "MQF = 0: %{customdata:.1f}%"
                "<extra></extra>"
            ),
        ),
        row=1,
        col=2,
    )

for phase_name in ["Baseline", "Long-term"]:
    phase_bins = vza_binned.loc[
        vza_binned["phase"] == phase_name
    ]

    fig.add_trace(
        go.Scatter(
            x=phase_bins["median_vza"],
            y=100 * phase_bins["median_anomaly"],
            mode="lines+markers",
            name=f"{phase_name} bin median",
            legendgroup=phase_name,
            line={
                "color": PHASE_COLORS[phase_name],
                "width": 3,
            },
            marker={
                "size": 9,
                "symbol": "diamond",
            },
            error_y={
                "type": "data",
                "symmetric": False,
                "array": 100 * phase_bins["error_plus"],
                "arrayminus": 100 * phase_bins["error_minus"],
                "color": PHASE_COLORS[phase_name],
                "thickness": 1.4,
                "width": 4,
            },
            customdata=phase_bins["n"].to_numpy(),
            hovertemplate=(
                "Median VZA: %{x:.2f}°<br>"
                "Median anomaly: %{y:.1f}%<br>"
                "Dates: %{customdata}"
                "<extra></extra>"
            ),
        ),
        row=1,
        col=2,
    )

fig.add_hline(
    y=0,
    line={
        "color": "#64748B",
        "width": 1.2,
        "dash": "dash",
    },
    row=1,
    col=2,
)

for definition in PHASE_WINDOWS:
    start_date = EVENT_DATE + pd.Timedelta(
        days=definition["start_day"]
    )

    end_date = EVENT_DATE + pd.Timedelta(
        days=definition["end_day"] + 1
    )

    fig.add_vrect(
        x0=start_date,
        x1=end_date,
        fillcolor=PHASE_FILL_COLORS[definition["phase"]],
        line_width=0,
        layer="below",
        row=1,
        col=1,
    )

fig.add_vline(
    x=EVENT_DATE,
    line={
        "color": HAIYAN_COLOR,
        "width": 2,
        "dash": "dash",
    },
    row=1,
    col=1,
)

fig.add_annotation(
    x=EVENT_DATE,
    y=1.02,
    xref="x",
    yref="paper",
    text="Haiyan/Yolanda",
    showarrow=False,
    xanchor="left",
    yanchor="bottom",
    font={
        "size": 13,
        "color": HAIYAN_COLOR,
    },
    bgcolor="rgba(255,255,255,0.85)",
)

fig.add_annotation(
    x=0.98,
    y=0.03,
    xref="x2 domain",
    yref="y3 domain",
    text=(
        f"Baseline Spearman ρ = {baseline_vza_spearman:.2f}<br>"
        f"Predicted 10° to 50° difference = "
        f"{predicted_10_to_50_change_pp:+.1f} pp"
    ),
    showarrow=False,
    xanchor="right",
    yanchor="bottom",
    align="right",
    font={
        "size": 13,
        "color": "#111827",
    },
    bgcolor="rgba(255,255,255,0.90)",
    borderpad=5,
)

fig.update_xaxes(
    title_text="Date",
    row=1,
    col=1,
)

fig.update_xaxes(
    title_text="Daily median sensor zenith angle (°)",
    range=[0, 70],
    row=1,
    col=2,
)

fig.update_yaxes(
    title_text="Sensor zenith angle (°)",
    range=[0, 70],
    row=1,
    col=1,
    secondary_y=False,
)

fig.update_yaxes(
    title_text="Standardised DNB-BRDF",
    row=1,
    col=1,
    secondary_y=True,
)

fig.update_yaxes(
    title_text="Baseline-relative anomaly (%)",
    range=[-110, 130],
    row=1,
    col=2,
)

fig.update_layout(
    title={
        "text": (
            "Viewing geometry and direct DNB-BRDF sensitivity "
            "over the fixed settlement-light mask"
        ),
        "x": 0.01,
        "xanchor": "left",
    },
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    font={
        "family": "Arial",
        "size": 16,
        "color": "#111827",
    },
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.10,
        "xanchor": "right",
        "x": 1,
    },
    margin={
        "l": 90,
        "r": 85,
        "t": 140,
        "b": 75,
    },
    width=1550,
    height=650,
)

fig.show()

fig.write_html(
    FIGURE_DIR / "05_vza_signal_sensitivity.html",
    include_plotlyjs="cdn",
)

### Figure 5. VZA Evolution and Radiance Sensitivity

Figure 5 evaluates whether viewing geometry may influence direct DNB-BRDF radiance over the fixed settlement-light mask. The temporal panel compares smoothed median VZA with standardised direct radiance, while the binned panel checks whether baseline-relative radiance anomalies change systematically across VZA ranges.

This diagnostic should be interpreted as a reliability test, not as a recovery result. If radiance anomalies vary strongly with VZA, then part of the apparent post-event signal may be geometry-sensitive. If the relationship is weak or inconsistent, VZA may still contribute noise but is less likely to dominate the phase-level interpretation.

The next step is to repeat key recovery metrics under all valid observations, VZA <= 40 degrees, and VZA <= 30 degrees. A metric should be flagged as geometry-sensitive if shock magnitude, cumulative deficit, T50, T80, or recovery slope changes materially under stricter VZA filtering.


In [ ]:
# %% Cell 16 — final phase-level observability decision table

phase_decision_summary = (
    phase_observability[
        [
            "phase",
            "days",
            "days_ge_90pct",
            "mqf_0_median",
            "mqf_0_q25",
            "mqf_0_q75",
            "no_retrieval_pct",
        ]
    ]
    .merge(
        phase_composite_summary,
        on="phase",
        how="left",
    )
)


def classify_phase_observability(row):
    if row["days_ge_90pct"] >= 3:
        return "Directly observable"

    if (
        row["mqf_0_median"] >= 50
        or row["days_ge_90pct"] >= 1
    ):
        return "Marginally observable"

    return "Observation-limited"


phase_decision_summary[
    "observability_status"
] = phase_decision_summary.apply(
    classify_phase_observability,
    axis=1,
)

phase_decision_summary[
    "interpretation_rule"
] = np.select(
    [
        (
            phase_decision_summary[
                "observability_status"
            ]
            == "Directly observable"
        ),
        (
            phase_decision_summary[
                "observability_status"
            ]
            == "Marginally observable"
        ),
    ],
    [
        (
            "Phase-level composite and broad radiance "
            "change may be interpreted."
        ),
        (
            "Interpret broad direction only; do not assign "
            "precise shock or recovery timing."
        ),
    ],
    default=(
        "Do not infer recovery status from direct "
        "DNB-BRDF for this phase."
    ),
)

phase_decision_summary.to_csv(
    TABLE_DIR / "phase_observability_decisions.csv",
    index=False,
)

display(
    phase_decision_summary.style.format(
        {
            "mqf_0_median": "{:.1f}",
            "mqf_0_q25": "{:.1f}",
            "mqf_0_q75": "{:.1f}",
            "no_retrieval_pct": "{:.1f}",
            "median_completeness_pct": "{:.1f}",
            "composite_median_radiance": "{:.3f}",
        },
        na_rep="—",
    )
)

print(
    "Completed outputs:\n"
    "1. 01_fixed_signal_mask.html\n"
    "2. 02_phase_90pct_dnb_composites.html\n"
    "3. 03_observed_vs_gap_filled.html\n"
    "4. 04_observability_daily_and_phase.html\n"
    "5. 05_vza_signal_sensitivity.html\n\n"
    "No shock, T50, or T80 metric has been inferred here. "
    "Those metrics should only be estimated after applying "
    "the phase-level observability decisions."
)

### Final Observability Decision Table

The final decision table is the gatekeeper for recovery interpretation. Under the current rule, phases with at least three dates above the high-completeness threshold can support phase-level composite interpretation. Because all phases meet that minimum, the Haiyan prototype is not completely observation-blocked at the phase scale.

However, this does not yet justify precise recovery timing. The diagnostics show that observability is intermittent, direct and gap-filled products can diverge, and VZA may need to be tested before fine-scale claims. Therefore, the defensible conclusion is:

Phase-level Haiyan diagnostics are observable enough for broad impact and recovery interpretation, but not yet sufficient for unqualified daily recovery metrics.

Shock magnitude, T50, T80, recovery slope, and duration below baseline should only be estimated after assigning each unit an observation status, product-sensitivity status, geometry-sensitivity status, and baseline-stability status.
